# Expedia Travel Experience Text Extraction and Analysis

## Project Overview

Travel marketplaces contain a mix of structured information, such as ratings and ticket prices, and unstructured text, such as attraction descriptions and customer reviews. This project turns Expedia attraction pages for New York into an analytical dataset, then uses that dataset to support a practical travel-planning decision.

The notebook follows the full data science workflow: data extraction, data cleaning, exploratory analysis, review-text analysis, interpretation, and final recommendations.

## Problem Statement

A traveler planning a short trip to New York has many highly rated attractions to choose from. The challenge is not only finding options with strong ratings, but also understanding price, popularity, provider coverage, cancellation flexibility, and what customers actually say in their reviews.

This project uses Expedia attraction listings and review text to answer a focused question: **Which New York attractions provide the strongest combination of quality, popularity, budget fit, and customer experience signals?**

## Business and Analytical Goal

The goal is to build a reproducible, reviewer-friendly text extraction and analysis workflow that helps convert web content into travel-planning insight.

The analysis answers these questions:

1. How can attraction and review text be extracted from Expedia pages into structured datasets?
2. What does the cleaned attraction dataset reveal about ratings, prices, review volume, and tour providers?
3. Which attractions are the most expensive, and does a $10,000 budget cover the top 10 most expensive options?
4. Among the most expensive attractions, which one has the strongest review-volume signal?
5. Are there full-day trips in the scraped attraction titles?
6. What themes appear in the extracted customer review text?
7. How should the data be used to plan a practical trip itinerary?

## 1. Environment Setup

This notebook uses the same tools as the original workflow: `pandas` for tabular analysis and Selenium for browser-based text extraction. The live scraper is optional because travel websites can block repeated automated access, and because a professional project should be reproducible from saved data snapshots.

In [ ]:
import time
import warnings

import pandas as pd

try:
    from selenium import webdriver
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.support.ui import WebDriverWait as wait
    from selenium.common.exceptions import NoSuchElementException
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)
pd.set_option("display.max_colwidth", 120)

## 2. Project Configuration

The notebook defaults to the saved CSV files so the analysis can run from top to bottom without repeatedly opening Expedia in a browser. Set `RUN_LIVE_SCRAPER = True` only when you intentionally want to collect fresh data from the website.

In [ ]:
RUN_LIVE_SCRAPER = False

SEARCH_URL = "https://www.expedia.com/things-to-do/search?%2Fthings-to-do%2Fsearch%3F=&challengeReferer=noref&d1=2025-10-28&d2=2025-10-29&endDate=2025-10-29&filter.reviewScore=GT4_5&filter.seeAll=true&location=New%20York%20%28and%20vicinity%29%2C%20New%20York%2C%20United%20States%20of%20America&regionType=&rid=178293&selectedId=&slimSearchKeyword=&sort=RECOMMENDED&startDate=2025-10-28&swp=on"

ATTRACTIONS_RAW_PATH = "attractions_raw.csv"
ATTRACTIONS_CLEAN_PATH = "attractions.csv"
REVIEWS_PATH = "reviews.csv"

BUDGET = 10000

## 3. Text Extraction Strategy

The extraction workflow has two stages.

First, the search-results page is opened and attraction links are collected. Second, each attraction page is opened and the scraper extracts page text for the attraction name, company, rating, number of reviews, ticket price, cancellation policy, overview, location, meeting point, and page URL.

A `for` loop is the right control structure for the attraction pages because the scraper already has a finite list of URLs. A `while` loop would be more appropriate for a scraper that clicks a "next" button until no more pages are available.

In [ ]:
def build_driver():
    """Create a Chrome driver with the same anti-automation settings used in the original scraper."""
    if not SELENIUM_AVAILABLE:
        raise ImportError("Selenium is not available in this environment. Use the saved CSV snapshots or install Selenium.")

    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.implicitly_wait(10)
    return driver

### 3.1 Collect Attraction Links

The live scraper can collect attraction URLs directly from Expedia. When the scraper is not running, the notebook uses the attraction page URLs already saved in `attractions_raw.csv`. This keeps the analysis reproducible and avoids unnecessary requests to Expedia.

In [ ]:
links = []

if RUN_LIVE_SCRAPER:
    driver = build_driver()
    driver.get(SEARCH_URL)
    time.sleep(30)

    try:
        elements = driver.find_elements(
            "xpath",
            '//div[@class="uitk-card uitk-card-roundcorner-all uitk-card-has-border uitk-layout-grid-item uitk-card-has-primary-theme"]/a'
        )
        for elem in elements:
            link = elem.get_attribute("href")
            if link:
                links.append(link)
    except Exception:
        links.append("No Link")
else:
    attraction_snapshot = pd.read_csv(ATTRACTIONS_RAW_PATH, index_col=0)
    links = attraction_snapshot["Attraction_page"].dropna().tolist()

print(f"Collected {len(links)} attraction links.")
links[:3]

### Link Collection Interpretation

The saved snapshot contains 75 attraction URLs. These URLs represent Expedia attractions in New York for the selected October 28-29, 2025 travel window, filtered to highly rated options. Keeping these links in the snapshot makes the downstream analysis traceable because each row can be connected back to its source page.

### 3.2 Extract Attraction Page Text

The helper functions below keep the original Selenium and XPath-based approach while removing repeated code. Each field is still extracted from the same type of page element, and each extraction step keeps a fallback value in case a page does not expose that field.

In [ ]:
def read_element_text(driver, xpath, default_value):
    """Read text from one page element and return a default value if the element is missing."""
    try:
        return driver.find_element("xpath", xpath).get_attribute("textContent").strip()
    except Exception:
        return default_value


def extract_attraction_details(driver, link):
    """Extract the attraction-level fields from one Expedia attraction page."""
    driver.get(link)
    time.sleep(10)

    return {
        "Attraction_name": read_element_text(
            driver,
            '//h4[@class="uitk-heading uitk-heading-4 uitk-type-style-headline-large"]',
            "No Name"
        ),
        "Company": read_element_text(
            driver,
            '//div[@class="uitk-text uitk-type-300 uitk-text-white-space-pre-line uitk-text-default-theme uitk-spacing uitk-spacing-margin-blockstart-two"]',
            "No Company"
        ),
        "Rating": read_element_text(
            driver,
            '(//span[@class="uitk-badge-base-text"])[1]',
            "No Ratings"
        ),
        "Reviews": read_element_text(
            driver,
            '//button[@class="uitk-link uitk-spacing uitk-spacing-margin-blockstart-one uitk-layout-flex-item uitk-link-align-left uitk-link-layout-default uitk-link-medium"]',
            "No Reviews"
        ),
        "Ticket_price": read_element_text(
            driver,
            '(//span[@class="uitk-lockup-price"])[1]',
            "No Price"
        ),
        "Free_cancellation": read_element_text(
            driver,
            '//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-layout-grid uitk-layout-grid-has-columns uitk-layout-grid-has-columns-by-medium uitk-layout-grid-has-columns-by-large uitk-spacing uitk-spacing-margin-blockstart-two"]/li[1]',
            "No Cancellation Details"
        ),
        "Overview": read_element_text(
            driver,
            '(//div[@class="uitk-layout-flex-item uitk-layout-flex-item-flex-grow-1"])[2]',
            "No Overview"
        ),
        "Location": read_element_text(
            driver,
            '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[1]/li[1]',
            "No Location"
        ),
        "Meeting_point": read_element_text(
            driver,
            '(//ul[@class="uitk-typelist uitk-typelist-orientation-stacked uitk-typelist-size-2 uitk-typelist-spacing"])[2]/li[1]',
            "No Meeting Point"
        ),
        "Attraction_page": link
    }

In [ ]:
attra_dic = {
    "Attraction_name": [],
    "Company": [],
    "Rating": [],
    "Reviews": [],
    "Ticket_price": [],
    "Free_cancellation": [],
    "Overview": [],
    "Location": [],
    "Meeting_point": [],
    "Attraction_page": []
}

if RUN_LIVE_SCRAPER:
    if "driver" not in globals():
        driver = build_driver()

    for start_index in range(0, min(len(links), 75), 25):
        print(f"Collecting attraction records {start_index} to {start_index + 24}...")
        for link in links[start_index:start_index + 25]:
            attraction_record = extract_attraction_details(driver, link)
            for column, value in attraction_record.items():
                attra_dic[column].append(value)

    attraction_data_raw = pd.DataFrame(attra_dic)
    attraction_data_raw.to_csv(ATTRACTIONS_RAW_PATH)
else:
    attraction_data_raw = pd.read_csv(ATTRACTIONS_RAW_PATH, index_col=0)

attraction_data_raw.head()

### Attraction Extraction Interpretation

The raw attraction dataset is intentionally text-heavy. Even fields that look numeric, such as ratings, review counts, and prices, are collected as page text first. Cleaning those text fields is required before they can be used for ranking, budgeting, or comparison.

## 4. Data Cleaning

The cleaning step converts the scraped text into analysis-ready columns:

- `Rating` becomes a numeric score.
- `Ticket_price` removes the dollar sign and becomes a numeric price.
- `Reviews` extracts the review count from the longer review-label text.
- `Company` removes the leading word `By` so provider names can be grouped cleanly.

In [ ]:
attractions_df = attraction_data_raw.copy()

attractions_df["Rating"] = pd.to_numeric(attractions_df["Rating"], errors="coerce")
attractions_df["Ticket_price"] = (
    attractions_df["Ticket_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace("No Price", "", regex=False)
)
attractions_df["Ticket_price"] = pd.to_numeric(attractions_df["Ticket_price"], errors="coerce")
attractions_df["Reviews"] = (
    attractions_df["Reviews"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")[0]
)
attractions_df["Reviews"] = pd.to_numeric(attractions_df["Reviews"], errors="coerce")
attractions_df["Company"] = (
    attractions_df["Company"]
    .astype(str)
    .str.replace(r"^By\s*", "", regex=True)
    .str.strip()
)

attractions_df.to_csv(ATTRACTIONS_CLEAN_PATH)
attractions_df.info()

### Cleaning Interpretation

The cleaned snapshot contains 75 attraction records and 10 columns. The key analytical fields are now numeric: `Rating`, `Reviews`, and `Ticket_price`. This matters because the project questions require ranking, filtering, grouping, and budget calculations.

### 4.1 Missing Values and Basic Validation

The original task asked to drop missing ticket prices and fill missing ratings with zero. In this snapshot, the cleaned price and rating columns do not contain missing values, but the same operations are included so the notebook remains valid if a future live scrape returns incomplete records.

In [ ]:
missing_summary = attractions_df[["Ticket_price", "Rating", "Reviews"]].isna().sum()
missing_summary

In [ ]:
attractions_df = attractions_df.dropna(subset=["Ticket_price"]).copy()
attractions_df["Rating"] = attractions_df["Rating"].fillna(0)

attractions_df[["Attraction_name", "Company", "Rating", "Reviews", "Ticket_price"]].head()

### Missing-Value Interpretation

Because ticket prices are complete in the saved data, dropping missing ticket prices does not reduce the dataset. Filling missing ratings also does not change this snapshot, but it protects the analysis from breaking if a later extraction returns an attraction without a visible rating.

## 5. Attraction-Level Analysis

This section turns the cleaned attraction table into travel-planning insight. The focus is on rating quality, price, popularity, provider coverage, and whether specific trip types can be found from the attraction titles.

### 5.1 Inspect the 15th Attraction Row

A row-level inspection is useful before broader analysis because it confirms that each record combines descriptive text, numeric values, and source-page metadata as expected.

In [ ]:
attractions_df.iloc[14]

### Row-Level Interpretation

The 15th row is `Circle Line: Statue of Liberty at Sunset Cruise`. It has a rating of 9.8, a ticket price of $32, and 6 reviews in the saved snapshot. This is a good example of the dataset structure: the row contains the attraction title, provider, numeric quality signals, price, location, meeting point, and the source URL.

### 5.2 Rating Range and Top-Rated Attractions

The source search was already filtered toward highly rated experiences, so the rating distribution should be narrow. This step confirms the range and identifies all attractions with ratings of 9.0 or higher.

In [ ]:
print("Rating range:", attractions_df["Rating"].min(), "to", attractions_df["Rating"].max())

highly_rated_attractions = attractions_df[attractions_df["Rating"] >= 9]
highly_rated_attractions[["Attraction_name", "Company", "Rating", "Reviews", "Ticket_price"]]

### Rating Interpretation

All 75 attractions meet the 9.0-or-higher threshold. That means rating alone is not enough to choose among the options. The next filters, especially review volume, price, and customer review themes, are needed to separate broadly validated experiences from newer or less-reviewed listings.

### 5.3 Search for Full-Day Trips in Attraction Titles

The title text is searched for the phrase `full-day`. This is a simple but useful form of text analysis because trip duration is often embedded directly in marketplace listing titles.

In [ ]:
full_day_attractions = attractions_df[
    attractions_df["Attraction_name"].str.contains("full-day", case=False, na=False)
]

full_day_attractions[["Attraction_name", "Company", "Rating", "Ticket_price"]]

### Full-Day Trip Interpretation

No attraction title in this snapshot contains the phrase `full-day`. There are day-oriented listings, such as Washington, D.C. day-trip options, but the exact `full-day` wording does not appear in the scraped titles. For itinerary planning, this means duration should not be inferred from title text alone; the attraction page details should be checked before scheduling.

### 5.4 Top 10 Most Expensive Attractions

Price is one of the clearest planning constraints. Sorting by `Ticket_price` identifies the attractions that would have the largest impact on a travel budget.

In [ ]:
expensive_10 = attractions_df.sort_values("Ticket_price", ascending=False).head(10)
expensive_10[["Attraction_name", "Company", "Rating", "Reviews", "Ticket_price", "Free_cancellation"]]

### Expensive Attraction Interpretation

The top of the price list is dominated by helicopter tours, premium carriage rides, Broadway tickets, and bundled sightseeing products. The most expensive listing is `The Empire Helicopter Tour of New York` at $429. High price does not always mean high review volume: several premium listings have very few reviews, while CityPASS and Broadway options show much stronger demand signals.

### 5.5 Budget Check for the Top 10 Most Expensive Attractions

The original planning scenario uses a $10,000 budget. This step checks whether the top 10 most expensive attractions would fit within that budget.

In [ ]:
total_cost = expensive_10["Ticket_price"].sum()
print("Total cost of top 10 attractions:", total_cost)

if total_cost <= BUDGET:
    print("Yes, a $10,000 budget will cover all top 10 attractions.")
else:
    print("No, a $10,000 budget is not enough for all top 10 attractions.")

### Budget Interpretation

The top 10 most expensive attractions cost $2,293 in total, so a $10,000 budget covers them comfortably. The budget question is therefore not whether the traveler can afford the activities; it is whether those high-cost activities are the best use of time and money compared with lower-cost attractions that have stronger review volume.

### 5.6 Most Reviewed Attraction Among the Top 10 Most Expensive

Review volume is a useful confidence signal. Among expensive attractions, the most-reviewed option is the one with the strongest evidence base from prior travelers.

In [ ]:
most_reviewed_expensive = expensive_10.sort_values(by="Reviews", ascending=False).head(1)
most_reviewed_expensive[["Attraction_name", "Company", "Rating", "Reviews", "Ticket_price"]]

### Review-Volume Interpretation

Among the top 10 most expensive attractions, `New York CityPASS: Experience 5 must-see attractions` has the most reviews, with 2,106 reviews and a $154 ticket price. This makes CityPASS a stronger evidence-backed option than several higher-priced premium listings with much lower review counts.

### 5.7 Travel Agency Coverage

Grouping attractions by provider shows whether the market is concentrated among a few agencies or spread across many one-off providers.

In [ ]:
agency_counts = (
    attractions_df
    .groupby("Company")["Attraction_name"]
    .count()
    .sort_values(ascending=False)
)

agency_counts

### Provider Interpretation

The provider landscape is fragmented. ExperienceFirst, Intrepid Travel, and City Wonders each appear 4 times, while New York Water Taxi, Unlimited Biking NYC, and Broadway Inbound US each appear 3 times. Most other providers appear only once or twice. For planning, this suggests there are several repeat providers worth comparing, but the dataset still contains many specialized experiences from individual vendors.

### 5.8 Data-Driven Trip Shortlist

The shortlist below applies the trip-planning logic described in the original notebook: prioritize strong ratings, meaningful review volume, reasonable prices, and cancellation flexibility.

In [ ]:
recommended_shortlist = attractions_df[
    (attractions_df["Rating"] >= 9.4) &
    (attractions_df["Reviews"] >= 50) &
    (attractions_df["Ticket_price"] <= 100) &
    (attractions_df["Free_cancellation"].str.contains("Free cancellation", case=False, na=False))
].sort_values(["Reviews", "Rating"], ascending=[False, False])

recommended_shortlist[["Attraction_name", "Company", "Rating", "Reviews", "Ticket_price", "Location"]].head(10)

### Shortlist Interpretation

This shortlist balances quality and practicality. It avoids relying only on the highest ratings, because a perfect 10.0 with very few reviews may be less reliable than a 9.4 or 9.6 with hundreds of reviews. For an actual itinerary, this filtered list would be combined with location and meeting-point information to reduce travel time between activities.

## 6. Customer Review Text Extraction

The second extraction task collects review text from the first two attraction pages. Review text adds a qualitative layer that rating scores cannot provide. A 10/10 score tells us the customer was satisfied; the text explains why.

In [ ]:
def read_element_list_text(elements, position, default_value=""):
    """Read one text item from a list of Selenium elements."""
    try:
        return elements[position].get_attribute("textContent").strip()
    except Exception:
        if elements:
            return elements[-1].get_attribute("textContent").strip()
        return default_value


def extract_reviews_for_link(driver, link):
    """Extract one visible page of review text from an Expedia attraction page."""
    driver.get(link)
    time.sleep(10)

    names = driver.find_elements(
        "xpath",
        '//div[@class="uitk-layout-grid-item"][2]/div/h3[@class="uitk-heading uitk-heading-6 uitk-spacing uitk-spacing-margin-blockstart-two"]'
    )
    scores = driver.find_elements(
        "xpath",
        '//div[@class="uitk-layout-grid-item"][2]/div/h4[@class="uitk-heading uitk-heading-6"]'
    )
    dates = driver.find_elements(
        "xpath",
        '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-text uitk-type-300 uitk-text-default-theme"][2]'
    )
    countries = driver.find_elements(
        "xpath",
        '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-text uitk-type-300 uitk-text-default-theme"][3]'
    )
    texts = driver.find_elements(
        "xpath",
        '//div[@class="uitk-layout-grid-item"][2]/div/div[@class="uitk-expando-peek uitk-spacing uitk-spacing-margin-blockstart-two"]//div[@class="uitk-text overflow-wrap uitk-type-300 uitk-text-default-theme"]'
    )

    records = []
    for index, text in enumerate(texts):
        records.append({
            "Attraction_page": link,
            "Customer_name": read_element_list_text(names, index, "No Name"),
            "Review_scores": read_element_list_text(scores, index),
            "Review_date": read_element_list_text(dates, index),
            "Country": read_element_list_text(countries, index),
            "Review_text": text.get_attribute("textContent").strip()
        })

    return records

In [ ]:
review_dic = {
    "Attraction_page": [],
    "Customer_name": [],
    "Review_scores": [],
    "Review_date": [],
    "Country": [],
    "Review_text": []
}

if RUN_LIVE_SCRAPER:
    if "driver" not in globals():
        driver = build_driver()

    review_records = []
    for link in links[:2]:
        review_records.extend(extract_reviews_for_link(driver, link))

    reviews_df = pd.DataFrame(review_records)
    reviews_df.to_csv(REVIEWS_PATH)
else:
    reviews_df = pd.read_csv(REVIEWS_PATH, index_col=0)

reviews_df.head()

### Review Extraction Interpretation

The saved review dataset contains 20 review rows: 10 from `SUMMIT One Vanderbilt Experience Tickets` and 10 from `Statue of Liberty & Ellis Island Tour: All Options`. This is a small but useful sample for demonstrating review-text extraction, cleaning, and interpretation.

## 7. Review Text Cleaning

Review text often contains encoded characters, punctuation artifacts, and extra whitespace. The cleaning step removes special characters while preserving readable words, numbers, spaces, and basic punctuation.

In [ ]:
reviews_df["Review_text"] = (
    reviews_df["Review_text"]
    .astype(str)
    .str.replace(r"&#\d+;", "", regex=True)
    .str.replace(r"[^A-Za-z0-9\s.,!?'"]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

reviews_df.to_csv(REVIEWS_PATH)
reviews_df.head()

### Text Cleaning Interpretation

The cleaned review text is easier to scan and analyze. Some original spelling and grammar issues remain because they are part of the customer-written text. The goal is not to rewrite the reviews, but to remove technical noise that would interfere with text analysis.

## 8. Review Text Analysis

This section uses the cleaned review text to identify patterns in customer language. The analysis stays intentionally simple and transparent: review counts, score extraction, word counts, frequent terms, and theme flags.

In [ ]:
page_to_attraction = attractions_df.set_index("Attraction_page")["Attraction_name"]

review_analysis = reviews_df.copy()
review_analysis["Attraction_name"] = review_analysis["Attraction_page"].map(page_to_attraction)
review_analysis["Review_score_numeric"] = (
    review_analysis["Review_scores"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)
review_analysis["Review_word_count"] = review_analysis["Review_text"].str.split().str.len()

review_summary = (
    review_analysis
    .groupby("Attraction_name")
    .agg(
        Review_count=("Review_text", "count"),
        Average_score=("Review_score_numeric", "mean"),
        Median_word_count=("Review_word_count", "median"),
        Longest_review_words=("Review_word_count", "max")
    )
    .sort_values("Review_count", ascending=False)
)

review_summary

### Review Summary Interpretation

Both reviewed attractions have 10 scraped reviews in this sample, and all scraped review scores are 10.0 out of 10. Because the scores do not vary, the text becomes the more useful source of insight. The review lengths also show that some customers leave short approval signals, while others provide operational details about guides, timing, crowding, and the experience itself.

### 8.1 Frequent Terms in Review Text

Frequent terms help identify what customers mention most often. This is not a full natural-language-processing model; it is a transparent text-frequency analysis using the cleaned review content.

In [ ]:
stop_words = [
    "the", "and", "for", "with", "was", "were", "are", "you", "your", "our", "that", "this",
    "from", "have", "had", "has", "but", "not", "all", "very", "much", "more", "than", "they",
    "their", "there", "into", "about", "what", "when", "where", "who", "why", "how", "really"
]

review_words = (
    review_analysis["Review_text"]
    .str.lower()
    .str.replace(r"[^a-z\s]", " ", regex=True)
    .str.split(expand=True)
    .stack()
)
review_words = review_words[review_words.str.len() > 2]
review_words = review_words[~review_words.isin(stop_words)]

top_review_terms = review_words.value_counts().head(20)
top_review_terms

### Frequent-Term Interpretation

The most frequent terms include words such as `great`, `tour`, `guide`, `amazing`, `experience`, `liberty`, `informative`, and `views`. These terms point to two different experience drivers: visual impact for SUMMIT One Vanderbilt, and guided storytelling or logistics for the Statue of Liberty and Ellis Island tour.

### 8.2 Review Theme Counts

The theme counts below use keyword patterns to summarize common review topics. This keeps the interpretation connected to the actual review text while avoiding unsupported claims from a small sample.

In [ ]:
theme_patterns = {
    "Experience/enjoyment": "great|amazing|fun|enjoyed|wonderful|recommended|excellent",
    "Guide/service quality": "guide|staff|friendly|knowledgeable|informative|organized|story",
    "Views/scenery": "view|views|city|spectacular",
    "Time/crowding/value concerns": "crowded|limited|time|line|too much|overrated",
    "Statue/Ellis Island context": "statue|liberty|ellis|island|ferry"
}

theme_counts = (
    pd.Series({
        theme: review_analysis["Review_text"].str.contains(pattern, case=False, na=False).sum()
        for theme, pattern in theme_patterns.items()
    })
    .sort_values(ascending=False)
    .rename("Review_count")
    .reset_index()
    .rename(columns={"index": "Theme"})
)

theme_counts

### Theme Interpretation

The strongest theme is positive experience language, appearing in 14 of 20 reviews. Guide and service quality appears in 10 reviews, mainly from the Statue of Liberty and Ellis Island tour. Time, crowding, or value concerns appear in 5 reviews, showing that even highly rated experiences can have friction points. Views and scenery appear in 3 reviews, concentrated around SUMMIT One Vanderbilt.

## 9. Answers to the Project Questions

1. **How was text extracted?** Attraction and review text were extracted with Selenium by opening Expedia pages, locating HTML elements with XPath, and reading their `textContent`. Saved CSV snapshots make the notebook reproducible without repeated scraping.
2. **What does the attraction dataset show?** The dataset contains 75 highly rated New York attractions. Ratings range from 9.0 to 10.0, so price and review volume are more useful differentiators than rating alone.
3. **Which attractions are most expensive?** The most expensive listing is `The Empire Helicopter Tour of New York` at $429. The top 10 expensive options are concentrated in helicopter tours, sports, Broadway, carriage rides, and bundled sightseeing products.
4. **Does a $10,000 budget cover the top 10 most expensive attractions?** Yes. The top 10 total is $2,293, leaving substantial room in the budget.
5. **Which expensive attraction has the most reviews?** Among the top 10 most expensive attractions, `New York CityPASS: Experience 5 must-see attractions` has the most reviews, with 2,106 reviews.
6. **Are there full-day trips in the titles?** No title contains the exact phrase `full-day` in this snapshot. Duration should be verified from each attraction page before scheduling.
7. **What do reviews reveal?** Review text is strongly positive, but it adds nuance beyond the scores. SUMMIT reviews emphasize views, visual experience, and crowding/value concerns. Statue of Liberty and Ellis Island reviews emphasize guides, organization, storytelling, and the need for enough time.

## 10. Trip Planning Recommendation

For a real trip plan, I would not simply buy the 10 most expensive attractions. I would prioritize attractions with a strong balance of rating, review volume, reasonable price, cancellation flexibility, and location fit.

A practical plan would start with evidence-backed, high-volume options such as CityPASS, Statue of Liberty and Ellis Island experiences, One World Observatory, MoMA, and selected Broadway or observation-deck experiences. Premium activities like helicopter tours can still be included, but they should be treated as optional splurges because several have high prices and relatively low review volume in this snapshot.

The review text also affects planning. SUMMIT One Vanderbilt appears visually memorable and fun, but some customers mention crowding or expectations around immersive features. The Statue of Liberty and Ellis Island tour benefits from guide quality, but reviewers also suggest allowing enough time. Those insights help plan not only what to book, but how to schedule the day.

## 11. Conclusion

This project demonstrates an end-to-end text extraction and text analysis workflow for travel planning. The scraper converts Expedia attraction and review pages into structured datasets, the cleaning process turns page text into usable analytical fields, and the analysis connects quantitative signals with customer-written review themes.

The main finding is that the scraped New York attraction set is uniformly highly rated, so the best planning decisions come from combining multiple signals: review volume, price, provider, cancellation flexibility, location, and review text. CityPASS stands out among expensive options because it has the strongest review-volume signal. SUMMIT One Vanderbilt and the Statue of Liberty and Ellis Island tour both receive very positive review scores, but their text reveals different strengths and planning considerations.

The main limitations are the small review sample, the high-rating filter in the original Expedia search, reliance on website structure that may change, and the lack of deeper NLP modeling because the project intentionally preserves the original package set. Future improvements could include collecting more review pages, expanding to more cities or dates, adding time-of-day and route optimization, and using richer text-analysis methods if additional libraries are allowed.